# Planning Pattern - Plan, Observe, Replan


<img src="https://www.dailydoseofds.com/content/images/2026/01/https-3a-2f-2fsubstack-post-media-s3-amazonaws-com-2fpublic-2fimages-2f643b6891-84f6-4672-aa1f-4724c5ad2d12_716x526-3.gif" alt="Planning pattern" width="500"/>

This notebook uses the Planning Pattern to analyze a staged mobile-forensics case about a missing-phone interval, a call record, WhatsApp activity, and a later network-status finding that forces the investigation path to change. Instead of asking one tool question at a time, Planning asks you to build an ordered investigation path, inspect what changed, and revise that path when new evidence exposes a missing dependency.

Important: Figure 1 shows a general planning architecture in which a planner can hand tasks to a ReAct-style executor. In this Lab 4 notebook, we use a simpler version of the pattern: we focus on building and revising the plan itself, first manually and then with `PlanningAgent`, which is the local agent that implements the planning workflow pattern used in this lab, rather than using a separate `ReactAgent` executor.

In Lab 4, the `Environment` is the staged forensic evidence package described in `02_case_overview.md`. The notebook first gives the model only a partial artifact bundle, then introduces a new observation so students can see why replanning matters.

You can think of the later-supplied network-status evidence as standing in for the kind of result a ReAct-style execution step might return to a planner, but in this simplified lab it is manually provided from the withheld artifact rather than produced by an actual `ReactAgent`.

In this lab, the goal is not to accept the first timeline the model suggests. The goal is to create an initial plan, test that plan against new observations, revise it when needed, and produce a final conclusion that stays inside the evidence.

This is the **fourth pattern lab** in the course. If you want to revisit the earlier lessons first:

* [Reflection Pattern](../lab1_reflection_pattern/03_lab_notebook.ipynb)
* [Tool Use Pattern](../lab2_tool_use_pattern/03b_lab_notebook.ipynb)
* [ReAct Pattern](../lab3_react_pattern/03b_lab_notebook.ipynb)


## Lab Question and Response Format

Use the staged artifacts to answer this planning question:

**What timeline would you build from the partial artifact bundle, and how should that timeline change when the network-status artifact is added later in the full artifact bundle?**

Keep the same four-part planning record visible across the notebook:

1. `initial investigation plan`
2. `observation-driven replan`
3. `reconstructed timeline and timing conclusion`
4. `evidence mapping and remaining uncertainty`

Suggested layout for the guided walkthrough:
- Under `initial investigation plan`, state the goal, ordered steps, evidence needed, and replanning triggers.
- Under `observation-driven replan`, explain what changed and how the ordered steps should be updated.
- Under `reconstructed timeline and timing conclusion`, list the key events in order and state what happened inside the incident window versus after it.
- Under `evidence mapping and remaining uncertainty`, cite which artifacts support the conclusion and what still cannot be confirmed.

Required manual sequence for this guided notebook:
`partial artifact bundle -> initial plan -> new observation -> revised plan -> final bounded timeline conclusion`

Important vocabulary for this lab:
- `plan`: an ordered set of investigation steps.
- `artifact bundle`: the group of records currently available to the model.
- `observation`: a new fact that changes what the plan should do next.
- `replan`: revise the investigation path after new evidence appears.
- `evidence-bounded answer`: an answer that says only what the observed records support.

Run the notebook from top to bottom. You are not expected to write new Python for the guided walkthrough; your main task is to explain why the plan changes when the evidence changes.


### Step 1: Set Up the Notebook


Run this setup cell first. It loads the lab configuration from `.env`, checks that you opened the notebook from the correct folder, connects to the model client, and points the notebook to the Lab 4 data files.

To run a code cell, click inside the cell and press `Shift+Enter`, or use the notebook's Run button. Run cells in order so each later cell has the variables and artifact bundles created by earlier cells.

You do not need to understand every Python line here. If this cell fails, fix the setup issue before moving on.


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

# This setup cell assumes you opened the notebook from this lab folder.
# It loads this lab's .env, adds src/ to the import path, and prepares the case data.
LAB_NAME = 'lab4_planning_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError('Could not find the Lab 4 data folder')

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)


### Step 2: Build the Partial and Full Artifact Bundles

This lab intentionally withholds the network-status artifact at first. The `partial_artifact_bundle` is what the model sees when it builds its initial plan. The `new_observation` is introduced later to force a replan, and the `full_artifact_bundle` combines both views for the final conclusion.

In the general planning figure, this later result could come back from a ReAct-style executor. In this notebook, `new_observation` plays that same role conceptually, but it is manually supplied from the withheld `network_status.csv` artifact.


In [ ]:
# Show only a short excerpt from each file so the notebook stays readable.
def excerpt(filename: str, n_lines: int = 8) -> str:
    lines = (data_dir / filename).read_text().strip().splitlines()
    return '\n'.join(lines[:n_lines])


# The initial bundle intentionally leaves out network_status.csv.
partial_artifact_bundle = f"""
artifact_manifest.json
{(data_dir / 'artifact_manifest.json').read_text()}

unlock_events.csv
{excerpt('unlock_events.csv')}

call_log.csv
{excerpt('call_log.csv')}

whatsapp_events.csv
{excerpt('whatsapp_events.csv')}

chain_of_custody.csv
{excerpt('chain_of_custody.csv')}
"""

# This simulates a newly discovered observation that arrives after the initial plan.
new_observation = f"""
network_status.csv
{excerpt('network_status.csv')}
"""

# The full bundle is used only after the plan has been revised.
full_artifact_bundle = partial_artifact_bundle + '\n\n' + new_observation


## Part 1: Walk the Planning Workflow Manually

In Part 1, you build an initial investigation plan from incomplete evidence, inspect what changed when a new artifact appears, revise the plan, and only then produce a final bounded timeline conclusion. This keeps the planning logic visible before you hand the same workflow to `PlanningAgent`.


### Step 3: Build the Initial Investigation Plan

This step asks the model to create an ordered investigation path from the partial artifact bundle. Because the network-status record is still hidden, the initial plan should include possible replanning triggers instead of pretending the first plan is complete.


In [ ]:
PLANNING_SYSTEM_PROMPT = """
You are a digital forensics planning assistant.
Build ordered investigation plans, update them when new observations expose missing dependencies, and avoid unsupported conclusions.
"""

case_question = (
    'What timeline would you build from the partial artifact bundle, ' 
    'and how should that timeline change when the network-status artifact is added later in the full artifact bundle?'
)

# Ask the model for an initial plan using only the partial artifact bundle.
# `replanning triggers` means: what future evidence or contradiction should make the current plan change.
initial_plan_request = (
    'Build an initial investigation plan for this case.\n'
    'Return a plan with: (1) investigation goal, (2) ordered steps, (3) evidence needed, ' 
    '(4) replanning triggers.\n\n'
    f'Task:\n{case_question}\n\n'
    f'Artifacts:\n{partial_artifact_bundle}'
)

# Run the model once to get the initial plan.
initial_plan = client.chat.completions.create(
    messages=[
        {'role': 'system', 'content': PLANNING_SYSTEM_PROMPT},
        {'role': 'user', 'content': initial_plan_request},
    ],
    model=MODEL,
).choices[0].message.content

display(Markdown('### Initial Investigation Plan\n\n' + initial_plan))


### Step 4: Revise the Plan After the New Observation

The network-status record changes whether WhatsApp delivery could have happened during the incident window. Instead of continuing the old plan unchanged, this step asks the model to explain what changed and how the ordered steps should be updated.


In [ ]:
# Use the new observation to update the current plan rather than starting over from scratch.
# `remaining replanning triggers` means: what future finding would still require another revision after this replan.
revised_plan_request = (
    'Revise the current investigation plan using the new observation below.\n'
    'Return a revised plan with: (1) what changed, (2) revised ordered steps, (3) updated evidence priorities, ' 
    '(4) remaining replanning triggers.\n\n'
    f'Original task:\n{case_question}\n\n'
    f'Current plan:\n{initial_plan}\n\n'
    f'New observation:\n{new_observation}'
)

revised_plan = client.chat.completions.create(
    messages=[
        {'role': 'system', 'content': PLANNING_SYSTEM_PROMPT},
        {'role': 'user', 'content': revised_plan_request},
    ],
    model=MODEL,
).choices[0].message.content

display(Markdown('### Observation-Driven Replan\n\n' + revised_plan))


### Step 5: Write the Final Timeline Conclusion

Now that the plan has been revised, use the full artifact bundle to write a bounded final report. A strong answer should separate in-window activity from anything that could only happen after reconnection.


In [ ]:
# Ask for a final report only after the initial plan has been updated with the new observation.
final_report_request = (
    'Using the revised plan and full artifact set, produce a short evidence-cited timeline conclusion.\n'
    'Return a report with: (1) reconstructed timeline, (2) timing conclusion, (3) evidence mapping, ' 
    '(4) what remains uncertain.\n\n'
    f'Revised plan:\n{revised_plan}\n\n'
    f'Artifacts:\n{full_artifact_bundle}'
)

final_timeline_conclusion = client.chat.completions.create(
    messages=[
        {'role': 'system', 'content': PLANNING_SYSTEM_PROMPT},
        {'role': 'user', 'content': final_report_request},
    ],
    model=MODEL,
).choices[0].message.content

display(Markdown('### Reconstructed Timeline and Timing Conclusion\n\n' + final_timeline_conclusion))


## Part 2: Run the Same Workflow with `PlanningAgent`

The packaged `PlanningAgent` exposes the same workflow more directly: build an initial plan, revise the plan with new observations, and then run the full planning pattern in one call. Compare its behavior with your manual walkthrough instead of assuming the packaged version is automatically better.


### Step 6: Create the `PlanningAgent`

This cell creates the packaged planning agent that wraps the same pattern you just inspected manually.


In [ ]:
from agentic_patterns.planning_pattern.planning_agent import PlanningAgent

planning_agent = PlanningAgent(client=client, model=MODEL)


### Step 7: Let `PlanningAgent` Build the Initial Plan

This run uses the same partial artifact bundle from Part 1. Compare the agent's plan with the manual initial plan you reviewed earlier.


In [ ]:
packaged_initial_plan = planning_agent.build_initial_plan(
    f'{case_question}\n\nArtifacts:\n{partial_artifact_bundle}'
)
display(Markdown('### PlanningAgent Initial Plan\n\n' + packaged_initial_plan))


### Step 8: Let `PlanningAgent` Revise the Plan

This run injects the new network observation and asks the packaged agent to update its current plan. Check whether it revises the timeline in a reasonable way.


In [ ]:
packaged_revised_plan = planning_agent.revise_plan(
    user_msg=f'{case_question}\n\nArtifacts:\n{partial_artifact_bundle}',
    current_plan=packaged_initial_plan,
    observations=new_observation,
)
display(Markdown('### PlanningAgent Revised Plan\n\n' + packaged_revised_plan))


### Step 9: Run the Full Planning Workflow

This final cell hands the packaged agent the full artifact bundle. Compare its final output with your manual conclusion: did it preserve the same evidence limits, or did it overstate what the records prove?


In [ ]:
packaged_final_report = planning_agent.run(user_msg=f'{case_question}\n\nArtifacts:\n{full_artifact_bundle}')
display(Markdown('### PlanningAgent Final Report\n\n' + packaged_final_report))


## Manual vs. Packaged Planning

`Part 1` made the planning loop explicit: build an initial plan, inspect what changed, revise the plan, and then write a final bounded conclusion. `Part 2` handed the same case to `PlanningAgent`, which wrapped that workflow for you. In both versions, the key learning goal is the same: a planning pattern should adapt when new observations change the path.
